# 🔬 APA (Adaptive Precision Ascension) — Forensic Observability & Root-Cause Analyzer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RedSafir/Adaptive-Precision-Ascension/blob/main/analyze_forensic.ipynb)

Notebook ini didesain khusus untuk menganalisis dan membongkar log forensik (**`*_forensic.jsonl`**) dari framework **Adaptive Precision Ascension (APA)**.

Berbeda dengan log training biasa yang hanya mencatat *"modul apa yang overflow"*, log forensik APA menjawab pertanyaan esensial:
> **"Saat eskalasi terjadi di suatu step, tensor spesifik apa (input, weight, output, grad_output, grad_weight, atau grad_input) yang menjadi biang keroknya (`culprit`), berapa dimensinya, bagaimana dinamika numeriknya (mean/std), dan mengalir dari modul mana sebelumnya?"**

--- 
### 📑 Modul Analisis yang Tersedia:
1. **Colab File Ingestion**: Upload log langsung di Google Colab atau baca dari folder lokal `result/`.
2. **Executive Summary & KPI**: Total eskalasi, rasio `OVERFLOW` vs `SILENT_UNDERFLOW`, dan timeline kejadian.
3. **Culprit Tensor Breakdown**: Analisis komponen tensor mana yang paling sering menjadi biang instabilitas.
4. **Arsitektural Heatmap & Escalation Ladder**: Pemetaan lapisan jaringan mana yang paling rentan eskalasi (`FP8 -> FP16 -> TF32`).
5. **Dinamika Numerik & Rentang Dinamis**: Analisis statistik (`mean`, `std`, rasio sinyal terhadap gradien) saat kegagalan terjadi.
6. **Graf Propagasi Aliran Upstream (Error Propagation Graph)**: Visualisasi keterkaitan modul sebelumnya (`preceding_module`) terhadap modul yang tereskalasi.
7. **Case File Inspector**: Pemeriksa kasus per event eskalasi dengan kartu diagnosis otomatis.

## 1. Setup Environment & Library Import
Sel ini memuat semua pustaka visualisasi (Matplotlib, Seaborn, NetworkX, Pandas, NumPy) dan mengonfigurasi gaya visualisasi publication-ready.

In [ ]:
import os
import json
import glob
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter, defaultdict

# Setup style visualisasi modern
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

print("✅ Library berhasil dimuat. Environment siap untuk analisis forensik.")

## 2. Ingesti Data Log Forensik (Colab & Local)
Jika dijalankan di Google Colab, Anda dapat langsung mengunggah file `*_forensic.jsonl` atau notebook akan otomatis memindai file yang ada di direktori lokal.

In [ ]:
# Deteksi apakah sedang berjalan di Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("💻 Terdeteksi di Google Colab. Menghubungkan ke Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive berhasil di-mount di /content/drive!")
    except Exception as e:
        print(f"ℹ️ Google Drive mount info: {e}")

# Pola pencarian file log forensik (Google Drive, folder result, root)
search_patterns = [
    '/content/drive/MyDrive/result/*forensic*.jsonl',
    '/content/drive/My Drive/result/*forensic*.jsonl',
    '/content/drive/MyDrive/*forensic*.jsonl',
    '/content/drive/My Drive/*forensic*.jsonl',
    'result/*forensic*.jsonl',
    '*forensic*.jsonl',
    'result/*.jsonl',
    '*.jsonl'
]

forensic_files = []
for pat in search_patterns:
    forensic_files.extend(glob.glob(pat))

# Deduplikasi sambil menjaga urutan
forensic_files = list(dict.fromkeys(forensic_files))

if IN_COLAB and not forensic_files:
    print("⚠️ File tidak ditemukan otomatis di Google Drive. Silakan upload manual:")
    from google.colab import files
    uploaded = files.upload()
    forensic_files = list(uploaded.keys())

print(f"📁 File log yang terdeteksi ({len(forensic_files)} file):")
for idx, f in enumerate(forensic_files):
    size_kb = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"  [{idx:2d}] {f:<40s} ({size_kb:6.1f} KB)")

# Pilih file yang ingin dianalisis (default: file pertama atau file dengan nama 'forensic')
selected_idx = 0
for i, f in enumerate(forensic_files):
    if 'forensic' in f.lower():
        selected_idx = i
        break

SELECTED_FORENSIC_FILE = forensic_files[selected_idx] if forensic_files else None
print(f"\n👉 File forensik aktif yang dipilih: {SELECTED_FORENSIC_FILE}")

## 3. Parser Forensik Komprehensif
Fungsi ini membedah setiap rekaman JSONL forensik ke dalam DataFrame terstruktur, mengekstrak:
- Metadata event (step, modul, alasan, level perubahan).
- Per-role amax values (`input_activation`, `weight`, `output`, `grad_output`, `grad_weight`, `grad_input`).
- Statistik tensor (`mean`, `std`) untuk setiap role.
- Dimensi bentuk tensor (`tensor_shape`) dan upstream provenance (`preceding_module_in_forward_order`).

In [ ]:
def parse_forensic_log(filepath):
    if not filepath or not os.path.exists(filepath):
        print(f"⚠️ File {filepath} tidak ditemukan.")
        return pd.DataFrame(), pd.DataFrame()

    events = []
    role_stats_records = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            # Skip non-forensic headers
            if 'module_name' not in rec and 'culprit_tensor_role' not in rec:
                continue

            step = rec.get('step')
            mod = rec.get('module_name')
            reason = rec.get('reason')
            culprit = rec.get('culprit_tensor_role')
            old_lvl = rec.get('level_before')
            new_lvl = rec.get('level_after')
            amax = rec.get('amax_value')
            thresh = rec.get('threshold_at_time')
            shape = rec.get('tensor_shape')
            dtype = rec.get('dtype_at_time')
            preceding = rec.get('preceding_module_in_forward_order')
            timestamp = rec.get('timestamp_utc')
            argmax_idx = rec.get('argmax_flat_index')
            underflow_ratio = rec.get('underflow_ratio')

            per_role = rec.get('per_role_amax') or {}
            per_stats = rec.get('per_role_stats') or {}

            shape_str = ' x '.join(map(str, shape)) if isinstance(shape, list) else str(shape)
            transition_str = f"{old_lvl} ➜ {new_lvl}"

            # Calculate relative threshold violation ratio
            thresh_ratio = (amax / thresh) if (thresh and thresh > 0 and not math.isinf(thresh)) else 0.0

            event_dict = {
                'event_id': len(events) + 1,
                'step': step,
                'module_name': mod,
                'reason': reason,
                'transition': transition_str,
                'level_before': old_lvl,
                'level_after': new_lvl,
                'culprit_role': culprit,
                'amax_trigger': amax,
                'threshold': thresh,
                'threshold_ratio': thresh_ratio,
                'tensor_shape': shape_str,
                'dtype': dtype,
                'preceding_module': preceding if preceding else '(None/First)',
                'timestamp_utc': timestamp,
                'argmax_index': argmax_idx,
                'underflow_ratio': underflow_ratio,
                # Per-role amax columns
                'amax_input': per_role.get('input_activation'),
                'amax_weight': per_role.get('weight'),
                'amax_output': per_role.get('output'),
                'amax_grad_out': per_role.get('grad_output'),
                'amax_grad_weight': per_role.get('grad_weight'),
                'amax_grad_in': per_role.get('grad_input'),
            }
            events.append(event_dict)

            # Expand per-role stats (mean, std)
            if isinstance(per_stats, dict):
                for role, st in per_stats.items():
                    if isinstance(st, dict):
                        role_stats_records.append({
                            'event_id': len(events),
                            'step': step,
                            'module_name': mod,
                            'role': role,
                            'mean': st.get('mean'),
                            'std': st.get('std'),
                            'amax': per_role.get(role)
                        })

    df_events = pd.DataFrame(events)
    df_stats = pd.DataFrame(role_stats_records)
    return df_events, df_stats

df_events, df_stats = parse_forensic_log(SELECTED_FORENSIC_FILE)
print(f"📊 Berhasil mem-parsing {len(df_events)} event eskalasi forensik!")
if not df_events.empty:
    display(df_events[['event_id', 'step', 'module_name', 'reason', 'transition', 'culprit_role', 'amax_trigger', 'tensor_shape']].head(10))

## 4. Executive KPI Dashboard & Escalation Timeline
Visualisasi metrik tingkat tinggi untuk memahami frekuensi eskalasi sepanjang proses training dan distribusi penyebab eskalasi.

In [ ]:
if not df_events.empty:
    total_events = len(df_events)
    unique_modules = df_events['module_name'].nunique()
    overflow_count = (df_events['reason'] == 'OVERFLOW').sum()
    underflow_count = (df_events['reason'] == 'SILENT_UNDERFLOW').sum()
    top_culprit = df_events['culprit_role'].mode().iloc[0] if not df_events['culprit_role'].dropna().empty else "N/A"

    print("=" * 65)
    print("             📊 APA FORENSIC EXECUTIVE SUMMARY")
    print("=" * 65)
    print(f"  • Total Eskalasi Terdeteksi : {total_events}")
    print(f"  • Modul Terdampak           : {unique_modules} sub-layer unik")
    print(f"  • Silent Underflow Events   : {underflow_count} ({underflow_count/total_events*100:.1f}%)")
    print(f"  • Hard Overflow Events     : {overflow_count} ({overflow_count/total_events*100:.1f}%)")
    print(f"  • Biang Kerok Utama         : '{top_culprit}'")
    print("=" * 65)

    # Visualisasi Timeline & Breakdown Alasan
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4.5))

    # 1. Timeline Histogram
    ax1.hist(df_events['step'], bins=min(25, max(5, total_events)), color='#3B82F6', edgecolor='black', alpha=0.8)
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Jumlah Eskalasi')
    ax1.set_title('Distribusi Waktu Terjadinya Eskalasi (Step)')
    ax1.grid(True, alpha=0.3)

    # 2. Donut Chart Penyebab
    reason_counts = df_events['reason'].value_counts()
    colors = ['#F59E0B' if r == 'SILENT_UNDERFLOW' else '#EF4444' for r in reason_counts.index]
    ax2.pie(reason_counts, labels=reason_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, wedgeprops=dict(width=0.45))
    ax2.set_title('Proporsi Alasan Eskalasi (Reason)')

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Data kosong atau belum ada log forensik.")

## 5. Culprit Tensor Role Analysis: Membongkar Komponen Lemah
Bagian ini menunjukkan **komponen tensor spesifik mana yang paling sering memicu instabilitas numerik** di dalam operasi linear:
- `input_activation`: Aktivasi masuk terlalu besar atau terlalu kecil.
- `weight`: Bobot master atau salinan kerja mengalami perubahan ekstrim.
- `output`: Hasil perkalian matriks melebihi kapasitas representasi FP8.
- `grad_output` / `grad_weight` / `grad_input`: Gradien backpropagation underflow atau overflow.

In [ ]:
if not df_events.empty and 'culprit_role' in df_events.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # 1. Bar Chart Frekuensi Culprit Role
    culprit_counts = df_events['culprit_role'].value_counts(dropna=False)
    role_colors = {
        'input_activation': '#3B82F6', 
        'output': '#EF4444', 
        'weight': '#10B981', 
        'grad_output': '#8B5CF6',
        'grad_weight': '#EC4899',
        'grad_input': '#F59E0B'
    }
    bar_c = [role_colors.get(r, '#6B7280') for r in culprit_counts.index]
    
    sns.barplot(x=culprit_counts.values, y=culprit_counts.index.astype(str), palette=bar_c, ax=ax1)
    ax1.set_xlabel('Frekuensi Menjadi Penyebab (Count)')
    ax1.set_ylabel('Peran Tensor (Culprit Role)')
    ax1.set_title('Frekuensi Peran Tensor Pemicu Eskalasi')
    for i, v in enumerate(culprit_counts.values):
        ax1.text(v + 0.1, i, str(v), va='center', fontweight='bold')

    # 2. Cross-tabulation: Culprit Role vs Reason
    cross_tab = pd.crosstab(df_events['culprit_role'].fillna('(Unknown)'), df_events['reason'])
    cross_tab.plot(kind='barh', stacked=True, color=['#EF4444', '#F59E0B'], ax=ax2)
    ax2.set_xlabel('Jumlah Kejadian')
    ax2.set_ylabel('Peran Tensor')
    ax2.set_title('Korelasi Culprit Role vs Alasan Eskalasi')
    ax2.legend(title='Alasan')

    plt.tight_layout()
    plt.show()
    
    # Rincian dimensi tensor yang paling sering bermasalah
    print("📐 Rincian Bentuk Dimensi Tensor (Shape) yang Paling Sering Terlibat:")
    shape_counts = df_events['tensor_shape'].value_counts().head(5)
    for sh, cnt in shape_counts.items():
        print(f"  • Shape: {sh:<25s} ➜ {cnt} kali terjadi")

## 6. Arsitektural Vulnerability Heatmap: Modul Mana yang Paling Rentan?
Memvisualisasikan sub-modul dalam arsitektur Transformer (Multi-Head Attention vs MLP) yang mengalami eskalasi presisi, serta jenjang eskalasinya (`FP8 -> FP16` atau `FP16 -> TF32`).

In [ ]:
if not df_events.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(5, df_events['module_name'].nunique() * 0.45)))

    # 1. Modul yang paling sering tereskalasi
    mod_counts = df_events['module_name'].value_counts()
    sns.barplot(x=mod_counts.values, y=mod_counts.index, color='#4F46E5', ax=ax1)
    ax1.set_xlabel('Jumlah Eskalasi')
    ax1.set_ylabel('Nama Modul')
    ax1.set_title('Sub-Modul Paling Sering Mengalami Eskalasi')
    for i, v in enumerate(mod_counts.values):
        ax1.text(v + 0.05, i, str(v), va='center', fontweight='bold')

    # 2. Jenjang Eskalasi per Modul (FP8->FP16 vs FP16->TF32)
    trans_tab = pd.crosstab(df_events['module_name'], df_events['transition'])
    color_dict = {'FP8 ➜ FP16': '#3B82F6', 'FP16 ➜ TF32': '#EF4444'}
    bar_colors = [color_dict.get(c, '#6B7280') for c in trans_tab.columns]
    trans_tab.plot(kind='barh', stacked=True, color=bar_colors, ax=ax2)
    ax2.set_xlabel('Jumlah Kejadian')
    ax2.set_ylabel('')
    ax2.set_title('Jenjang Transisi Presisi per Sub-Modul')
    ax2.legend(title='Transisi')

    plt.tight_layout()
    plt.show()

## 7. Dinamika Numerik & Analisis Statistik Tensor (`mean` vs `std`)
Menganalisis besaran nilai rata-rata (`mean`) dan dispersi ragam (`std`) dari setiap role tensor saat momen eskalasi terjadi. Grafik ini membuktikan apakah layer mengalami *gradient vanishing* (gradien mendekati 0 sehingga silent underflow) atau *activation explosion* (aktivasi membengkak melebihi rentang FP8).

In [ ]:
if not df_stats.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # 1. Log-scale Standard Deviation per Role
    # Std mengindikasikan variansi nilai numerik tensor
    df_stats['std_safe'] = df_stats['std'].apply(lambda x: max(x, 1e-12) if pd.notnull(x) else np.nan)
    
    sns.boxplot(data=df_stats, x='role', y='std_safe', palette='Set2', ax=ax1)
    ax1.set_yscale('log')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=25, ha='right')
    ax1.set_xlabel('Peran Tensor (Role)')
    ax1.set_ylabel('Standar Deviasi (Log Scale)')
    ax1.set_title('Sebaran Variabilitas Numerik (Std) per Role Tensor')
    ax1.grid(True, which="both", ls="--", alpha=0.3)

    # 2. Amax per Role saat Insiden Terjadi
    df_stats['amax_safe'] = df_stats['amax'].apply(lambda x: max(x, 1e-12) if pd.notnull(x) else np.nan)
    sns.stripplot(data=df_stats, x='role', y='amax_safe', jitter=0.2, alpha=0.7, palette='tab10', ax=ax2)
    ax2.set_yscale('log')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=25, ha='right')
    ax2.set_xlabel('Peran Tensor (Role)')
    ax2.set_ylabel('Nilai Amax Absolut (Log Scale)')
    ax2.set_title('Nilai Puncak (Amax) Tiap Role Saat Eskalasi Terjadi')
    ax2.axhline(448.0, color='red', linestyle='--', alpha=0.7, label='FP8 E4M3 Max (448.0)')
    ax2.axhline(0.015625, color='orange', linestyle=':', alpha=0.7, label='FP8 E4M3 Min Positif (0.015625)')
    ax2.legend()
    ax2.grid(True, which="both", ls="--", alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Data statistik per-role (mean/std) tidak ditemukan dalam log.")

## 8. Graf Propagasi Aliran Upstream (Error Flow Graph)
Menelusuri dari mana instabilitas numerik mengalir menggunakan informasi `preceding_module_in_forward_order` $\rightarrow$ `module_name`. Ini memvisualisasikan bagaimana error merambat secara berantai antar modul.

In [ ]:
if not df_events.empty and 'preceding_module' in df_events.columns:
    # Bangun graf berarah
    G = nx.DiGraph()
    flow_pairs = df_events[df_events['preceding_module'] != '(None/First)'][['preceding_module', 'module_name']]
    pair_counts = flow_pairs.value_counts()

    if not pair_counts.empty:
        for (src, dst), weight in pair_counts.items():
            # Sederhanakan nama agar graf mudah dibaca
            src_clean = src.replace('blocks.', 'B').replace('.mha.', '_MHA_').replace('.mlp.', '_MLP_')
            dst_clean = dst.replace('blocks.', 'B').replace('.mha.', '_MHA_').replace('.mlp.', '_MLP_')
            G.add_edge(src_clean, dst_clean, weight=weight)

        plt.figure(figsize=(12, 7))
        pos = nx.spring_layout(G, k=1.2, seed=42)
        node_degrees = dict(G.degree())
        node_sizes = [max(800, node_degrees[n] * 600) for n in G.nodes()]
        edge_widths = [max(1.5, G[u][v]['weight'] * 1.2) for u, v in G.edges()]

        nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='#60A5FA', edgecolors='#1E3A8A', alpha=0.9)
        nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='#DC2626', arrowsize=20, arrowstyle='->', alpha=0.8)
        nx.draw_networkx_labels(G, pos, font_size=9, font_family='sans-serif', font_weight='bold')

        # Edge labels showing transition frequency
        edge_labels = {(u, v): f"{d['weight']}x" for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9, font_color='#991B1B')

        plt.title('Rantai Propagasi Aliran Numerik Upstream ➜ Modul Eskalasi', fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("ℹ️ Tidak ada data keterkaitan upstream antar modul.")

## 9. Case File Inspector: Bedah Insiden Eskalasi per Event
Fungsi interaktif di bawah ini mencetak berkas investigasi (*case file*) lengkap untuk suatu insiden eskalasi tertentu, termasuk diagnosis otomatis dan rekomendasi teknis.

In [ ]:
def inspect_forensic_case(event_number=1):
    if df_events.empty:
        print("Data kosong.")
        return

    idx = max(1, min(event_number, len(df_events))) - 1
    row = df_events.iloc[idx]

    print("=" * 75)
    print(f" 🕵️ CASE FILE #{row['event_id']} — Eskalasi pada Step {row['step']}")
    print("=" * 75)
    print(f"  • Target Modul        : {row['module_name']}")
    print(f"  • Alasan Eskalasi     : {row['reason']}")
    print(f"  • Transisi Presisi    : {row['transition']} (Dtype awal: {row['dtype']})")
    print(f"  • Modul Upstream      : {row['preceding_module']}")
    print(f"  • Dimensi Tensor      : {row['tensor_shape']}")
    print(f"  • Culprit Tensor Role : 🚨 {row['culprit_role'].upper() if pd.notnull(row['culprit_role']) else 'UNKNOWN'}")
    print(f"  • Trigger Metric      : {row['amax_trigger']:.6f} (Batas Representasi: {row['threshold']})")
    print(f"  • Timestamp (UTC)     : {row['timestamp_utc']}")
    print("-" * 75)
    print("  📊 Kondisi Amax Setiap Role Tensor Saat Kejadian:")
    roles = [
        ('Input Activation', row['amax_input']),
        ('Weight', row['amax_weight']),
        ('Output Activation', row['amax_output']),
        ('Gradient Output', row['amax_grad_out']),
        ('Gradient Weight', row['amax_grad_weight']),
        ('Gradient Input', row['amax_grad_in']),
    ]
    for label, val in roles:
        val_str = f"{val:.6f}" if pd.notnull(val) else "N/A (Tidak tercatat step ini)"
        is_culprit = " 🚨 [CULPRIT]" if pd.notnull(row['culprit_role']) and label.lower().replace(' ', '_') in row['culprit_role'].lower() else ""
        print(f"    • {label:<22s}: {val_str}{is_culprit}")
    print("-" * 75)
    
    # Diagnosis otomatis
    print("  💡 DIAGNOSIS TEKNIS & REKOMENDASI:")
    if row['reason'] == 'SILENT_UNDERFLOW':
        print("    » Terjadi akumulasi gradien yang terlalu kecil (< 0.015625) melebihi batas toleransi underflow theta.")
        print("    » Tindakan: Eskalasi otomatis ke FP16/TF32 sudah tepat untuk mencegah gradient vanishing pada layer ini.")
    elif row['reason'] == 'OVERFLOW':
        print("    » Terjadi aktivasi atau bobot yang melebihi batas kapasitas maksimum representasi FP8 (403.2).")
        print("    » Tindakan: Batch saat ini otomatis di-skip untuk melindungi integritas bobot FP32 master.")
    print("=" * 75)

# Jalankan inspeksi pada Event pertama
inspect_forensic_case(1)

Gunakan slider atau ubah nomor event di bawah untuk memeriksa rekaman kejadian lainnya:

In [ ]:
# Ubah angka di dalam fungsi untuk memeriksa event nomor lain (misal: event nomor 2, 5, dst.)
inspect_forensic_case(2)

## 10. Ekspor Data Forensik ke CSV
Simpan ringkasan investigasi forensik ke format CSV untuk pelaporan, visualisasi eksternal, atau lampiran jurnal penelitian.

In [ ]:
if not df_events.empty:
    csv_filename = 'apa_forensic_summary_report.csv'
    df_events.to_csv(csv_filename, index=False)
    print(f"✅ Laporan ringkasan forensik berhasil diekspor ke: {csv_filename}")
    
    if IN_COLAB:
        from google.colab import files
        print("Unduh file laporan CSV langsung ke komputer Anda:")
        files.download(csv_filename)
else:
    print("Data kosong, tidak ada file CSV yang diekspor.")